# cl100k_base: Armenian Token Coverage

Scan the entire cl100k_base vocabulary (OpenAI's tokenizer for text-embedding-3-large)
to check how many tokens contain Armenian characters.

Source: [openai/tiktoken](https://github.com/openai/tiktoken) — the vocabulary is published openly.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
print(f"Encoding: {enc.name}")
print(f"Vocab size: {enc.n_vocab:,}")

## Scan vocabulary for Armenian characters

Armenian Unicode ranges:
- U+0530–U+058F (Armenian block)
- U+FB00–U+FB17 (Armenian ligatures in Alphabetic Presentation Forms)

In [ ]:
armenian_range = set(range(0x0530, 0x0590)) | set(range(0xFB00, 0xFB18))

armenian_tokens = []
for token_id in range(enc.n_vocab):
    try:
        token_bytes = enc.decode_single_token_bytes(token_id)
        try:
            text = token_bytes.decode("utf-8")
            if any(ord(c) in armenian_range for c in text):
                armenian_tokens.append((token_id, token_bytes, text))
        except UnicodeDecodeError:
            pass
    except Exception:
        pass

print(f"Total vocab size: {enc.n_vocab:,}")
print(f"Tokens containing Armenian characters: {len(armenian_tokens)}")

if armenian_tokens:
    print("\nArmenian-containing tokens:")
    for tid, tbytes, text in armenian_tokens[:50]:
        print(f"  id={tid:>6}: {tbytes!r:>30} -> '{text}'  ({len(text)} chars, {len(tbytes)} bytes)")
else:
    print("\nZERO tokens in cl100k_base contain Armenian characters.")
    print("Every Armenian character is split into individual bytes.")

## Compare with other scripts

For context: how many tokens does cl100k_base have for Russian (Cyrillic) and Georgian (another Caucasian script)?

In [ ]:
scripts = {
    "Armenian (U+0530-U+058F)": set(range(0x0530, 0x0590)),
    "Georgian (U+10A0-U+10FF)": set(range(0x10A0, 0x1100)),
    "Cyrillic (U+0400-U+04FF)": set(range(0x0400, 0x0500)),
    "Latin (U+0041-U+007A)": set(range(0x0041, 0x007B)),
    "Arabic (U+0600-U+06FF)": set(range(0x0600, 0x0700)),
    "Devanagari (U+0900-U+097F)": set(range(0x0900, 0x0980)),
}

for script_name, char_range in scripts.items():
    count = 0
    for token_id in range(enc.n_vocab):
        try:
            token_bytes = enc.decode_single_token_bytes(token_id)
            text = token_bytes.decode("utf-8")
            if any(ord(c) in char_range for c in text):
                count += 1
        except Exception:
            pass
    print(f"{script_name}: {count:,} tokens")